# Sustainable Development Goal 11: Sustainable Cities and Communities

This Jupyter Notebook focuses on the development and preparation of the SDG 11 Sub-Index, which relates to the United Nations Sustainable Development Goal 11: Good Health and Well-Being.

The purpose of this notebook is to import, inspect, clean and prepare datasets associated with sustainable cities and communities indicators for use in a larger composite indicator project. The processed data will later contribute towards calculating a comparative SDG readiness score across countries.

The following datasets are obtained through: https://unstats.un.org/sdgs/dataportal/ 

**Disclaimer: This dataset has already been standardised by the UN SDG database, therefore some common data cleaning steps such as formatting corrections, unit conversions and major transformations were not required. The cleaning process mainly focused on handling missing values, identifying duplicates, selecting relevant indicators and preparing the data for analysis.**

### 1.1 Import Data

In [30]:
# Import pandas
import pandas as pd

# Load datasets
slum_df = pd.read_excel("EN_LND_SLUM.xlsx")
disaster_df = pd.read_excel("VC_DSR_DAFF.xlsx")

# Display first 5 rows of each dataframe
print("Urban Slum Dataset:")
display(slum_df.head())

print("\nDisaster Affected Dataset:")
display(disaster_df.head())

PermissionError: [Errno 13] Permission denied: 'VC_DSR_DAFF.xlsx'

#### Meaning of Datasets:

- slum_df : Proportion of urban population living in slums (%) EN_LND_SLUM
- disaster_df : Number of directly affected persons attributed to disasters per 100,000 population (number) VC_DSR_DAFF

#### 2.1 Select Columns

In [ ]:
# Select columns for each dataset

# Years
years_2000_2024 = [str(year) for year in range(2000, 2025)]
years_2005_2025 = [str(year) for year in range(2005, 2026)]

# Clean column names
slum_df.columns = slum_df.columns.str.strip()
disaster_df.columns = disaster_df.columns.str.strip()

# slum_df columns
slum_df = slum_df[['GeoAreaName'] + years_2000_2024]

# disaster_df columns
disaster_df = disaster_df[['GeoAreaName'] + years_2005_2025]

# Display first 5 rows
print("Slum Dataset:")
display(slum_df.head())

print("\nDisaster Dataset:")
display(disaster_df.head())

Slum Dataset:


,GeoAreaName,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Afghanistan,NaN,NaN,NaN,NaN,NaN,NaN,63.60000,NaN,63.60000,...,NaN,72.12863,NaN,73.30000,NaN,73.30000,NaN,71.58904,NaN,NaN
1,Albania,28.1,NaN,25.5,NaN,23.00000,NaN,20.50000,NaN,17.90000,...,NaN,7.80000,NaN,5.30000,NaN,2.80000,NaN,2.70000,NaN,NaN
2,Algeria,NaN,NaN,NaN,NaN,NaN,NaN,30.80000,NaN,30.80000,...,NaN,21.07581,NaN,17.17042,NaN,13.26502,NaN,13.24600,NaN,NaN
3,Andorra,0.0,NaN,0.0,NaN,0.00000,NaN,0.00000,NaN,0.00000,...,NaN,0.00000,NaN,0.00000,NaN,0.00000,NaN,0.00000,0.0,0.0
4,Angola,19.7,NaN,19.7,NaN,19.66039,NaN,25.79191,NaN,31.92344,...,NaN,56.44953,NaN,62.58105,NaN,62.60000,NaN,62.70000,NaN,NaN



Disaster Dataset:


,GeoAreaName,2005,2006,2007,2008,2009,2010,2011,2012,2013,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Afghanistan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,69.052215,24.728493,765.247448,NaN,NaN,NaN,NaN,NaN,NaN
1,Albania,0.357589,0.098335,0.066162,0.100236,0.135068,34.998201,7.144074,0.687284,0.447108,...,5.279745,0.172518,161.804638,19.999931,NaN,NaN,4796.574923,817.004585,471.207283,NaN
2,Algeria,0.732241,16.769092,1.038333,88.339129,3.279756,2.633452,7.498044,12.212451,48.116076,...,2.232519,34.901043,58.796101,3.624013,237.341138,299.195067,143.642811,53.441823,95.039320,NaN
3,Andorra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,244.879786,0.000000,NaN
4,Angola,54.351317,81.807503,208.262738,455.088605,88.740582,80.339732,774.239304,78.697581,46.285928,...,257.992048,128.318858,106.003245,1239.651476,240.544924,0.622603,109.120158,1.567351,10.209617,0.06916


#### 2.2 Selection of Columns

At this stage, only the variables required for the **SDG 11 sub-index analysis** are retained from each dataset.

The selection keeps:

* all countries available within the datasets,
* identifying columns such as `GeoAreaName`
* yearly health indicator values from 2000 up to the most recent year available within each dataset.

### 3.1 Dealing with Duplicates

Duplicates will be dealt in 4.1 Check & Handling of Missing Values (Imputation of Missing Data)

In [ ]:
# Check duplicates in both datasets

for name, df in [('Slum Dataset', slum_df), ('Disaster Dataset', disaster_df)]:

    duplicates = df.duplicated(subset=['GeoAreaName']).sum()

    if duplicates > 0:
        print(f"{name}: {duplicates} duplicate countries found")
    else:
        print(f"{name}: No duplicate countries found")

Slum Dataset: No duplicate countries found
Disaster Dataset: 332 duplicate countries found


#### slum_df, disaster_df

For the SDG 11 datasets, I only kept the columns that were needed for the analysis. These included:

- GeoAreaName
- Year columns from 2000 up to the latest available year in the dataset

The remaining columns were removed as they were not necessary for building the SDG 11 sub-index.

The datasets are mainly organised by country, where each row represents a country and its values across multiple years. Because of this, the GeoAreaName column will be used as the main linking field when merging datasets together later in the project. This allows indicators from different SDG 11 datasets to align correctly based on the country.


### 4.1 Check & Handling of Missing Values (Imputation of Missing Data)

#### Slums Dataset

In [ ]:
# Check missing/NaN values per country in slum_df

missing_per_country = slum_df.isnull().sum(axis=1)

# Keep only countries with missing values
missing_per_country = missing_per_country[missing_per_country > 0]

# Create dataframe showing countries and missing value counts
missing_values_df = pd.DataFrame({
    'GeoAreaName': slum_df.loc[missing_per_country.index, 'GeoAreaName'],
    'Missing Values': missing_per_country.values
})

display(missing_values_df)

,GeoAreaName,Missing Values
0,Afghanistan,16
1,Albania,13
2,Algeria,16
3,Andorra,11
4,Angola,13
...,...,...
213,Western Sahara,13
214,World,2
215,Yemen,13
216,Zambia,13


In [ ]:
# ---------------------------------------------
# Remove countries with too many missing values
# ---------------------------------------------

slum_df['Missing_Count'] = slum_df[years_2000_2024].isnull().sum(axis=1)

# Remove rows with more than 18 missing values
slum_df = slum_df[
    slum_df['Missing_Count'] <= 18
].copy()

slum_df.drop(columns='Missing_Count', inplace=True)

# -------------------
# Fill missing values
# -------------------

for index, row in slum_df.iterrows():

    # Convert yearly values to numeric series
    values = row[years_2000_2024].astype(float)

    # Loop through each year
    for i in range(len(years_2000_2024)):

        year = years_2000_2024[i]

        # Skip if value already exists
        if pd.notnull(values[year]):
            continue

        # -----------------------------------------------------------------
        # Missing value between two existing values (use neighbour average)
        # -----------------------------------------------------------------
        if i > 0 and i < len(years_2000_2024) - 1:

            prev_year = years_2000_2024[i - 1]
            next_year = years_2000_2024[i + 1]

            prev_value = values[prev_year]
            next_value = values[next_year]

            if pd.notnull(prev_value) and pd.notnull(next_value):

                # Linear average
                estimated_value = (prev_value + next_value) / 2

                slum_df.at[index, year] = estimated_value
                continue

        # -------------------------------------------------------------------
        # Missing values at the start (use trend from nearest values)
        # -------------------------------------------------------------------
        if i == 0:

            next_valid = values.dropna()

            if len(next_valid) >= 2:

                first = next_valid.iloc[0]
                second = next_valid.iloc[1]

                # Calculate trend
                trend = second - first

                # Estimate backwards
                estimated_value = first - trend

                slum_df.at[index, year] = estimated_value
                continue

        # ------------------------------------------------------------------
        # Missing values at the end (use trend from previous values)
        # ------------------------------------------------------------------
        if i == len(years_2000_2024) - 1:

            prev_valid = values.dropna()

            if len(prev_valid) >= 2:

                last = prev_valid.iloc[-1]
                second_last = prev_valid.iloc[-2]

                # Calculate trend
                trend = last - second_last

                # Estimate forward
                estimated_value = last + trend

                slum_df.at[index, year] = estimated_value
                continue

# -----------------------------------------------------------------------
# Interpolate remaining gaps (Handles consecutive missing values)
# -----------------------------------------------------------------------
slum_df[years_2000_2024] = slum_df[
    years_2000_2024
].interpolate(
    axis=1,
    limit_direction='both'
)

display(slum_df)

remaining_missing = slum_df[years_2000_2024].isnull().sum().sum()

print(f"Remaining Missing Values: {remaining_missing}")

,GeoAreaName,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Afghanistan,63.60000,63.600000,63.60000,63.600000,63.60000,63.600000,63.60000,63.600000,63.60000,...,70.908460,72.12863,72.714315,73.30000,73.300000,73.30000,72.444520,71.58904,70.733560,69.87808
1,Albania,28.10000,26.800000,25.50000,24.250000,23.00000,21.750000,20.50000,19.200000,17.90000,...,9.050000,7.80000,6.550000,5.30000,4.050000,2.80000,2.750000,2.70000,2.650000,2.60000
2,Algeria,30.80000,30.800000,30.80000,30.800000,30.80000,30.800000,30.80000,30.800000,30.80000,...,23.028510,21.07581,19.123115,17.17042,15.217720,13.26502,13.255510,13.24600,13.236490,13.22698
3,Andorra,0.00000,0.000000,0.00000,0.000000,0.00000,0.000000,0.00000,0.000000,0.00000,...,0.000000,0.00000,0.000000,0.00000,0.000000,0.00000,0.000000,0.00000,0.000000,0.00000
4,Angola,19.70000,19.700000,19.70000,19.680195,19.66039,22.726150,25.79191,28.857675,31.92344,...,53.383770,56.44953,59.515290,62.58105,62.590525,62.60000,62.650000,62.70000,62.750000,62.80000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
213,Western Sahara,39.14517,42.203365,45.26156,43.428230,41.59490,38.381355,35.16781,33.075950,30.98409,...,18.746065,17.23863,16.810455,16.38228,20.031550,23.68082,20.131175,16.58153,13.031885,9.48224
214,World,31.19870,30.934170,30.91582,30.537470,30.11461,29.652990,29.21797,28.664270,28.17293,...,24.952990,24.61492,24.548890,24.36166,24.171970,24.20500,24.702350,24.76384,24.794585,24.82533
215,Yemen,63.81216,62.501400,61.19064,59.879875,58.56911,57.258345,55.94758,54.636815,53.32605,...,44.830735,44.20000,44.200000,44.20000,44.200000,44.20000,44.200000,44.20000,44.200000,44.20000
216,Zambia,63.78455,63.008230,62.23191,61.455585,60.67926,59.902940,59.12662,58.350300,57.57398,...,52.139730,51.36341,50.587090,49.81077,49.034450,48.25813,48.258130,48.25813,48.258130,48.25813


Remaining Missing Values: 0


Upon closer inspection of the dataset, I noticed that many of the missing values occurred between existing yearly data rather than entire sections being missing. Because of this, I decided that most of the missing data should not simply be removed. Instead, where a missing value appeared between two existing years, I filled it using the average of the neighbouring values. In my opinion, this was the most reasonable approach because it follows the overall trend already present within the country’s data.

I also decided to completely remove countries that contained more than 18 missing values. The gaps within these records were too large, meaning there was not enough reliable surrounding data to estimate missing values accurately. Keeping these rows would have introduced too much uncertainty into the analysis.

For countries with consecutive missing values, I used interpolation to estimate the missing data based on the surrounding trend. This method helped preserve the general direction of the data, whether the indicator was increasing or decreasing over time, while still keeping the dataset as consistent and realistic as possible for later analysis.


#### Disaster Dataset

In [ ]:
disaster_df = disaster_df[['GeoAreaName'] + years_2005_2025]

# Remove identical duplicate rows
disaster_df = disaster_df.drop_duplicates()

# Keep only first occurrence of duplicated countries
disaster_df = disaster_df.drop_duplicates(
    subset='GeoAreaName',
    keep='first'
)

# -----------------------------
# Check NaN and blank missing values per country
# -----------------------------

missing_per_country = (
    disaster_df.isnull().sum(axis=1) +
    (disaster_df == '').sum(axis=1)
)

# Keep only countries with missing values
missing_per_country = missing_per_country[
    missing_per_country > 0
]

# Create dataframe
missing_values_df = pd.DataFrame({
    'GeoAreaName': disaster_df.loc[
        missing_per_country.index,
        'GeoAreaName'
    ],
    'Missing Values': missing_per_country.values
})

display(missing_values_df)

# -----------------------------
# Remaining duplicate check
# -----------------------------
duplicate_count = disaster_df.duplicated(
    subset='GeoAreaName'
).sum()

print(f"Remaining Duplicate Countries: {duplicate_count}")
print(f"Countries With Missing Values: {len(missing_values_df)}")

KeyError: "['2000', '2001', '2002', '2003', '2004'] not in index"

Before handling missing values, I decided to remove duplicate rows from the dataset. During inspection, I noticed that some countries appeared more than once with exactly the same yearly data. In my opinion, these were simply repeated copies of the same record rather than separate observations.

Because the duplicated rows contained identical information, I kept only the first occurrence of each country and removed the rest. This helped keep the dataset consistent and prevented duplicates from affecting the later analysis and imputation process.

In [ ]:
# ---------------------------------------------
# Remove countries with too many missing values
# ---------------------------------------------

disaster_df['Missing_Count'] = disaster_df[years_2005_2025].isnull().sum(axis=1)

# Remove rows with more than 18 missing values
disaster_df = disaster_df[
    disaster_df['Missing_Count'] <= 18
].copy()

disaster_df.drop(columns='Missing_Count', inplace=True)

# -------------------
# Fill missing values
# -------------------

for index, row in disaster_df.iterrows():

    # Convert yearly values to numeric series
    values = row[years_2005_2025].astype(float)

    # Loop through each year
    for i in range(len(years_2005_2025)):

        year = years_2005_2025[i]

        # Skip if value already exists
        if pd.notnull(values[year]):
            continue

        # Missing value between two existing values
        if i > 0 and i < len(years_2005_2025) - 1:

            prev_year = years_2005_2025[i - 1]
            next_year = years_2005_2025[i + 1]

            prev_value = values[prev_year]
            next_value = values[next_year]

            if pd.notnull(prev_value) and pd.notnull(next_value):

                estimated_value = (prev_value + next_value) / 2

                disaster_df.at[index, year] = estimated_value
                continue

        # Missing values at the start
        if i == 0:

            next_valid = values.dropna()

            if len(next_valid) >= 2:

                first = next_valid.iloc[0]
                second = next_valid.iloc[1]

                trend = second - first
                estimated_value = first - trend

                disaster_df.at[index, year] = estimated_value
                continue

        # Missing values at the end
        if i == len(years_2005_2025) - 1:

            prev_valid = values.dropna()

            if len(prev_valid) >= 2:

                last = prev_valid.iloc[-1]
                second_last = prev_valid.iloc[-2]

                trend = last - second_last
                estimated_value = last + trend

                disaster_df.at[index, year] = estimated_value
                continue

# -----------------------------------------------------------------------
# Interpolate remaining gaps
# -----------------------------------------------------------------------

disaster_df[years_2005_2025] = disaster_df[
    years_2005_2025
].interpolate(
    axis=1,
    limit_direction='both'
)

display(disaster_df)

remaining_missing = disaster_df[years_2005_2025].isnull().sum().sum()

print(f"Remaining Missing Values: {remaining_missing}")

KeyError: "['2000', '2001', '2002', '2003', '2004'] not in index"